In [ ]:
%pip install -q isaacus python-dotenv pypdf python-docx beautifulsoup4 lxml ipywidgets

In [ ]:
from isaacus import Isaacus
from dotenv import load_dotenv
import os

from custom_taxonomy import CUSTOM_TAXONOMY
from renderer import make_ingest_file, build_taxonomy_app
from IPython.display import display

In [ ]:
load_dotenv()
isaacus_client = Isaacus(api_key=os.getenv("ISAACUS_API_KEY"))

## Taxonomy metadata viewer

Upload legal documents, enrich them with ILGS, classify them against the custom taxonomy, and review the extracted metadata in the interactive renderer.

In [ ]:
def clean_text(text):
    if text is None:
        return None
    value = str(text).replace("\n", " ").strip()
    return value or None


def decode_span(span, text):
    if not span:
        return None
    try:
        value = span.decode(text)
    except Exception:
        return None
    return clean_text(value)


def extract_date(doc):
    creation_dates = []
    other_dates = []
    for date in getattr(doc, "dates", []) or []:
        value = clean_text(getattr(date, "value", None))
        if not value:
            continue
        if getattr(date, "type", None) == "creation":
            creation_dates.append(value)
        else:
            other_dates.append(value)
    if creation_dates:
        return creation_dates[0]
    if other_dates:
        return other_dates[0]
    return None


def extract_title(doc):
    return decode_span(getattr(doc, "title", None), doc.text)


def extract_parties(doc):
    parties = []
    location_by_id = {loc.id: loc for loc in (getattr(doc, "locations", []) or []) if getattr(loc, "id", None)}
    for person in getattr(doc, "persons", []) or []:
        if getattr(person, "parent", None) is not None:
            continue
        role = getattr(person, "role", None)
        if role == "non_party":
            continue
        name = decode_span(getattr(person, "name", None), doc.text)
        if not name:
            continue
        party = {
            "name": name,
            "role": clean_text(str(role).replace("_", " ")) if role else None,
        }
        residence_id = getattr(person, "residence", None)
        if residence_id and residence_id in location_by_id:
            residence_name = decode_span(getattr(location_by_id[residence_id], "name", None), doc.text)
            if residence_name:
                party["residence"] = residence_name
        parties.append(party)
    return parties


def extract_locations(doc):
    values = []
    seen = set()
    for location in getattr(doc, "locations", []) or []:
        name = decode_span(getattr(location, "name", None), doc.text)
        if not name or name in seen:
            continue
        seen.add(name)
        values.append(name)
    return values


def extract_terms(doc):
    values = []
    for term in getattr(doc, "terms", []) or []:
        name = decode_span(getattr(term, "name", None), doc.text)
        if not name:
            continue
        values.append({
            "name": name,
            "definition": decode_span(getattr(term, "meaning", None), doc.text),
        })
    return values


def extract_signatures(doc):
    values = []
    for segment in getattr(doc, "segments", []) or []:
        if getattr(segment, "type", None) != "signature":
            continue
        body = decode_span(getattr(segment, "span", None), doc.text)
        title = decode_span(getattr(segment, "title", None), doc.text)
        if not title and body:
            words = body.split()
            title = " ".join(words[:6]) + ("…" if len(words) > 6 else "")
        if body or title:
            values.append({"title": title, "text": body})
    return values


def extract_enriched_features_from_document(document_text):
    response = isaacus_client.enrichments.create(
        model="kanon-2-enricher",
        texts=[document_text],
        overflow_strategy="auto",
    )
    doc = response.results[0].document
    features = {
        "title": extract_title(doc),
        "parties": extract_parties(doc),
        "date": extract_date(doc),
        "locations": extract_locations(doc),
        "terms": extract_terms(doc),
        "signatures": extract_signatures(doc),
    }
    return features, doc

In [ ]:
def score_document_by_category(document_text, query):
    result = isaacus_client.rerankings.create(
        model="kanon-2-reranker",
        query=query,
        texts=[document_text],
    )
    return result.results[0].score


def flatten_taxonomy(nodes):
    all_nodes = []
    for node in nodes:
        all_nodes.append(node)
        children = node.get("children", [])
        if children:
            all_nodes.extend(flatten_taxonomy(children))
    return all_nodes


def classify_greedy(document, nodes):
    scored_nodes = [
        {"node": node, "score": score_document_by_category(document, node["description"])}
        for node in nodes
    ]
    best = max(scored_nodes, key=lambda item: item["score"])
    best_node = best["node"]
    if not best_node.get("children"):
        return {
            "label": best_node["name"],
            "score": best["score"],
            "description": best_node["description"],
            "mode": "greedy",
        }
    return classify_greedy(document, best_node["children"])


def classify_full(document, nodes):
    all_nodes = flatten_taxonomy(nodes)
    scored_nodes = [
        {"node": node, "score": score_document_by_category(document, node["description"])}
        for node in all_nodes
    ]
    best = max(scored_nodes, key=lambda item: item["score"])
    best_node = best["node"]
    return {
        "label": best_node["name"],
        "score": best["score"],
        "description": best_node["description"],
        "mode": "full",
    }


def classify_document(document, nodes, mode="full"):
    if mode == "greedy":
        return classify_greedy(document, nodes)
    return classify_full(document, nodes)

In [ ]:
def pretty_title(label, parties, date):
    label = clean_text(label) or "Untitled"
    year = None
    if date:
        year = str(date).split("-")[0]

    first_party = None
    if parties:
        first = parties[0]
        if isinstance(first, dict):
            first_party = clean_text(first.get("name"))
        else:
            first_party = clean_text(first)

    if first_party and year:
        return f"{label} - {first_party} ({year})"
    if first_party:
        return f"{label} - {first_party}"
    if year:
        return f"{label} ({year})"
    return label

In [ ]:
ingest_file = make_ingest_file(
    extract_enriched_features_from_document,
    classify_document,
    CUSTOM_TAXONOMY,
    pretty_title,
)

app = build_taxonomy_app(
    ingest_file,
    taxonomy_mode="full",
    viewer_height="92vh",
    viewer_min_height="980px",
    viewer_max_height="1320px",
)
display(app)
